In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("⚠️  No GPU detected. Training will be slow. Consider using Google Colab (Runtime > Change runtime type > T4 GPU).")

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

#Load & Inspect Dataset

In [ ]:
raw_dataset = load_dataset("ag_news")
print(raw_dataset)

label_names = raw_dataset["train"].features["label"].names
print("Labels:", label_names)

sample = raw_dataset["train"][0]
print(f"\nSample text: {sample['text']}")
print(f"Label: {label_names[sample['label']]}")

In [ ]:
N_TRAIN   = 8000
N_TEST    = 1600
MAX_LEN   = 128
EPOCHS    = 5
BATCH     = 16
MODEL_NAME = "bert-base-uncased"

def balanced_sample(dataset, n_total, seed=42):
    n_per_class = n_total // 4
    subsets = []
    for label_id in range(4):
        class_subset = dataset.filter(lambda x: x["label"] == label_id)
        class_subset = class_subset.shuffle(seed=seed).select(range(n_per_class))
        subsets.append(class_subset)
    from datasets import concatenate_datasets
    combined = concatenate_datasets(subsets).shuffle(seed=seed)
    return combined

dataset = balanced_sample(raw_dataset["train"], N_TRAIN)
test_ds = balanced_sample(raw_dataset["test"],  N_TEST)

split    = dataset.train_test_split(test_size=0.2, seed=42)
train_ds = split["train"]
val_ds   = split["test"]

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

#Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding=False,
        truncation=True,
        max_length=MAX_LEN,
    )

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
val_tok   = val_ds.map(tokenize,   batched=True, remove_columns=["text"])
test_tok  = test_ds.map(tokenize,  batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenization complete.")
print("Features:", train_tok.features)

#Load Pre-trained BERT

In [ ]:
num_labels = len(label_names)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label={i: l for i, l in enumerate(label_names)},
    label2id={l: i for i, l in enumerate(label_names)},
)
print(f"Model loaded: {MODEL_NAME} with {num_labels} output classes")

#Define Metrics

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric       = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)
    f1  = f1_metric.compute(predictions=preds, references=labels, average="weighted")
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

#Training Arguments

In [ ]:
steps_per_epoch = len(train_ds) // BATCH
warmup_steps    = int(0.1 * steps_per_epoch * EPOCHS)

training_args = TrainingArguments(
    output_dir                  = "./bert-ag-news",
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH,
    per_device_eval_batch_size  = BATCH * 2,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    warmup_steps                = warmup_steps,
    label_smoothing_factor      = 0.1,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    greater_is_better           = True,
    logging_steps               = 50,
    fp16                        = torch.cuda.is_available(),
    dataloader_pin_memory       = torch.cuda.is_available(),
    report_to                   = "none",
    push_to_hub                 = False,
)
print(f"Warmup steps: {warmup_steps}")

#Fine-tune with Trainer

In [ ]:
trainer = Trainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_tok,
    eval_dataset     = val_tok,
    processing_class = tokenizer,
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

#Evaluate on Test Set

In [ ]:
results = trainer.evaluate(test_tok)
print("\n Test Set Results:")
for k, v in results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

#Detailed Classification Report

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

predictions = trainer.predict(test_tok)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred,
    display_labels=label_names,
    cmap="Blues",
    ax=ax
)
plt.title("Confusion Matrix — AG News (BERT)")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=120)
plt.show()

#Save Model

In [ ]:
trainer.save_model("./bert-ag-news-final")
tokenizer.save_pretrained("./bert-ag-news-final")
print("Model saved to ./bert-ag-news-final")

#Gradio Live Demo


In [ ]:
import gradio as gr
from transformers import pipeline
import torch

# Initialize the pipeline
classifier = pipeline(
    "text-classification",
    model="./bert-ag-news-final",
    tokenizer="./bert-ag-news-final",
    device=0 if torch.cuda.is_available() else -1,
    top_k=None,
)

EMOJI = {"World": "🌍", "Sports": "⚽", "Business": "💼", "Sci/Tech": "🔬"}

def predict(text):
    if not text.strip():
        return {}
    results = classifier(text[:512])[0]
    return {
        f"{EMOJI.get(r['label'], r['label'])}": round(r["score"], 4)
        for r in sorted(results, key=lambda x: -x["score"])
    }

examples = [
    ["NASA launches new Mars rover to study the planet's geology and search for signs of ancient life."],
    ["Stock markets surge as Federal Reserve signals potential interest rate cuts next quarter."],
    ["Brazil defeats Argentina 2-1 in the Copa América final after extra time."],
    ["United Nations holds emergency summit to address escalating tensions in the Middle East."],
]

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Paste a news headline or article snippet here...",
        label="News Text",
    ),
    outputs=gr.Label(num_top_classes=4, label="Category Probabilities"),
    title="📰 News Topic Classifier (BERT)",
    description="Classifies news into: World 🌍 · Sports ⚽ · Business 💼 · Sci/Tech 🔬",
    examples=examples,
    theme=gr.themes.Soft(),
)

demo.launch(
    share=True,
    debug=False,
    show_error=True
)